# Delta Lake com Apache Spark — Cenário E-commerce

## Fonte de dados
Dados simulados de um sistema de e-commerce com três entidades:
- **clientes** — cadastro de clientes da loja
- **produtos** — catálogo de produtos disponíveis
- **pedidos** — registro de compras realizadas

## Modelo ER
*(inserir imagem do modelo ER aqui)*

## Por que Delta Lake?
Delta Lake adiciona transações ACID, versionamento e time travel
sobre arquivos Parquet no Apache Spark.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# Iniciar SparkSession com Delta Lake
builder = SparkSession.builder \
    .appName("Delta Lake - E-commerce") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession iniciada com sucesso!")

your 131072x1 screen size is bogus. expect trouble
26/04/16 17:12:00 WARN Utils: Your hostname, DESKTOP-K76SSH3 resolves to a loopback address: 127.0.1.1; using 172.31.228.33 instead (on interface eth0)
26/04/16 17:12:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/bruno/apache-spark/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/bruno/.ivy2/cache
The jars for the packages stored in: /home/bruno/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-40157513-d22e-4187-af68-c77227476128;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 289ms :: artifacts dl 14ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0  

SparkSession iniciada com sucesso!


In [2]:
# DDL - Schema das tabelas (equivalente ao CREATE TABLE)

schema_clientes = StructType([
    StructField("id_cliente",    IntegerType(), False),
    StructField("nome",          StringType(),  False),
    StructField("email",         StringType(),  False),
    StructField("cidade",        StringType(),  True),
    StructField("data_cadastro", DateType(),    True),
    StructField("ativo",         BooleanType(), True),
])

schema_produtos = StructType([
    StructField("id_produto",    IntegerType(), False),
    StructField("nome_produto",  StringType(),  False),
    StructField("categoria",     StringType(),  True),
    StructField("preco",         DoubleType(),  True),
    StructField("estoque",       IntegerType(), True),
    StructField("ativo",         BooleanType(), True),
])

schema_pedidos = StructType([
    StructField("id_pedido",        IntegerType(), False),
    StructField("id_cliente",       IntegerType(), False),
    StructField("id_produto",       IntegerType(), False),
    StructField("data_pedido",      DateType(),    True),
    StructField("quantidade",       IntegerType(), True),
    StructField("valor_total",      DoubleType(),  True),
    StructField("status",           StringType(),  True),
    StructField("data_atualizacao", DateType(),    True),
])

print("Schemas definidos com sucesso!")

Schemas definidos com sucesso!


In [3]:
from datetime import date

# --- Caminhos das tabelas Delta ---
PATH_CLIENTES = "/tmp/delta/clientes"
PATH_PRODUTOS  = "/tmp/delta/produtos"
PATH_PEDIDOS   = "/tmp/delta/pedidos"

# --- INSERT clientes ---
dados_clientes = [
    (1, "Ana Silva",    "ana@email.com",    "São Paulo",       date(2023, 1, 10), True),
    (2, "Bruno Costa",  "bruno@email.com",  "Rio de Janeiro",  date(2023, 3, 22), True),
    (3, "Carla Mendes", "carla@email.com",  "Curitiba",        date(2023, 5, 15), True),
    (4, "Diego Lima",   "diego@email.com",  "Florianópolis",   date(2023, 7, 30), True),
]
df_clientes = spark.createDataFrame(dados_clientes, schema_clientes)
df_clientes.write.format("delta").mode("overwrite").save(PATH_CLIENTES)
print("✅ Tabela clientes criada")
df_clientes.show()

# --- INSERT produtos ---
dados_produtos = [
    (1, "Notebook Dell",    "Eletrônicos", 3500.00, 10, True),
    (2, "Mouse Logitech",   "Periféricos",   89.90, 50, True),
    (3, "Teclado Mecânico", "Periféricos",  299.00, 30, True),
    (4, "Monitor 24\"",     "Eletrônicos", 1200.00, 15, True),
]
df_produtos = spark.createDataFrame(dados_produtos, schema_produtos)
df_produtos.write.format("delta").mode("overwrite").save(PATH_PRODUTOS)
print("✅ Tabela produtos criada")
df_produtos.show()

# --- INSERT pedidos ---
dados_pedidos = [
    (1, 1, 2, date(2024, 1, 5),  2,   179.80, "entregue",   date(2024, 1, 7)),
    (2, 2, 1, date(2024, 1, 10), 1,  3500.00, "entregue",   date(2024, 1, 15)),
    (3, 3, 3, date(2024, 2, 3),  1,   299.00, "processando",date(2024, 2, 3)),
    (4, 1, 4, date(2024, 2, 20), 1,  1200.00, "enviado",    date(2024, 2, 21)),
]
df_pedidos = spark.createDataFrame(dados_pedidos, schema_pedidos)
df_pedidos.write.format("delta").mode("overwrite").save(PATH_PEDIDOS)
print("✅ Tabela pedidos criada")
df_pedidos.show()

Traceback (most recent call last):
  File "/home/bruno/apache-spark/.venv/lib/python3.14/site-packages/pyspark/serializers.py", line 459, in dumps
    return cloudpickle.dumps(obj, pickle_protocol)
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bruno/apache-spark/.venv/lib/python3.14/site-packages/pyspark/cloudpickle/cloudpickle_fast.py", line 73, in dumps
    cp.dump(obj)
    ~~~~~~~^^^^^
  File "/home/bruno/apache-spark/.venv/lib/python3.14/site-packages/pyspark/cloudpickle/cloudpickle_fast.py", line 632, in dump
    return Pickler.dump(self, obj)
           ~~~~~~~~~~~~^^^^^^^^^^^
  File "/home/bruno/apache-spark/.venv/lib/python3.14/site-packages/pyspark/cloudpickle/cloudpickle_fast.py", line 737, in reducer_override
    return self._function_reduce(obj)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/home/bruno/apache-spark/.venv/lib/python3.14/site-packages/pyspark/cloudpickle/cloudpickle_fast.py", line 592, in _function_reduce
    if _should_pickle_by_reference(

PicklingError: Could not serialize object: RecursionError: Stack overflow (used 8152 kB) while calling a Python object

In [4]:
# UPDATE — atualizar status de pedido e preço de produto

# Atualizar status do pedido 3 para "entregue"
dt_pedidos = DeltaTable.forPath(spark, PATH_PEDIDOS)
dt_pedidos.update(
    condition = col("id_pedido") == 3,
    set = {
        "status":           lit("entregue"),
        "data_atualizacao": lit(date(2024, 2, 10))
    }
)
print("✅ UPDATE — pedido 3 atualizado para 'entregue'")
spark.read.format("delta").load(PATH_PEDIDOS).show()

# Atualizar preço do produto 2
dt_produtos = DeltaTable.forPath(spark, PATH_PRODUTOS)
dt_produtos.update(
    condition = col("id_produto") == 2,
    set = { "preco": lit(79.90) }
)
print("✅ UPDATE — preço do Mouse atualizado para R$ 79.90")
spark.read.format("delta").load(PATH_PRODUTOS).show()

AnalysisException: [DELTA_MISSING_DELTA_TABLE] `/tmp/delta/pedidos` is not a Delta table.

In [ ]:
# DELETE — remover cliente inativo e produto descontinuado

# Desativar cliente 4 (soft delete — boa prática)
dt_clientes = DeltaTable.forPath(spark, PATH_CLIENTES)
dt_clientes.update(
    condition = col("id_cliente") == 4,
    set = { "ativo": lit(False) }
)
print("✅ Soft DELETE — cliente 4 desativado")
spark.read.format("delta").load(PATH_CLIENTES).show()

# Hard DELETE — remover pedidos cancelados (exemplo didático)
dt_pedidos.delete(condition = col("status") == "cancelado")
print("✅ Hard DELETE — pedidos cancelados removidos")
spark.read.format("delta").load(PATH_PEDIDOS).show()

In [ ]:
# TIME TRAVEL — consultar versões anteriores da tabela

# Ver histórico completo de alterações
print("📜 Histórico da tabela pedidos:")
dt_pedidos.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

# Ler versão 0 (antes dos updates)
print("🕐 Tabela pedidos — versão 0 (estado inicial):")
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(PATH_PEDIDOS) \
    .show()

# Ler versão atual
print("🕐 Tabela pedidos — versão atual:")
spark.read.format("delta").load(PATH_PEDIDOS).show()